# Hyperparameter-Optimierung (Server): RF, LSTM, XGBoost, LightGBM, CNN1D

**Wichtig:** Die Optimierung läuft *immer auf dem Server*.  
Pro Modell gibt es zwei **Deployment-Profile** (edge/server), die nur den **Suchraum** steuern – nicht den Ort der Optimierung.

- Zielvariable: `Group4-2_S6_VolumetricFlowRate`  
- `lags` & `horizon` sind **fix** (nicht Teil der Optimierung).  
- Zeitspalten werden als Features ausgeschlossen.  
- **Configs** (`config_*.json`) werden aus dem **gleichen Ordner** wie dieses Notebook geladen (und bei Bedarf erzeugt).

**Hinweis:** Dieses Notebook ersetzt die zwei vorherigen Notebooks und vereint alles hier.


In [ ]:

import os, json, math, random, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from pathlib import Path
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 120)

# ---- Settings ----
DATA_PATH = r"""C:\Users\ericg\Documents\Mechatronik M Sc\6. Semster\MA\Dev_Ma\ML_Edge_Device\Input\Input_Data\mqtt_data_rate_limited.csv"""
TARGET_COL = "Group4-2_S6_VolumetricFlowRate"
EXCLUDE_COLS = ["time", "datetime", "recording_timestamp", "duration"]

# Zeitreihen-Parameter (nicht optimiert)
LAGS = 4
HORIZON = 4

# Splits
TRAIN_FRACTION = 0.8
VAL_FRACTION = 0.2  # nur für DL-Modelle


In [ ]:

# ---- Portable display shim (kein externes Paket notwendig) ----
def display_dataframe_to_user(name, df):
    try:
        fn = f"{name}.csv".replace(" ", "_")
        df.to_csv(fn, index=False)
        print(f"[Saved] {fn}")
    except Exception as e:
        print("Could not save CSV:", e)
    try:
        display(df.head(20))
    except Exception:
        print(df.head(20))


## 1) Daten laden & Feature-Analyse

In [ ]:

# --- Load ---
df = pd.read_csv(DATA_PATH)
df.columns = [c.strip() for c in df.columns]
if "datetime" in df.columns:
    try:
        df["datetime"] = pd.to_datetime(df["datetime"])
        df = df.sort_values("datetime").set_index("datetime")
    except Exception:
        pass

print("Shape:", df.shape)
display(df.head(3))

# --- Missing ---
nulls = df.isna().mean().sort_values(ascending=False)
print("\nMissing-Rate (Top 10):")
display(nulls.head(10).to_frame("missing_rate"))

# --- Features ---
num_cols = df.select_dtypes(include=[np.number]).columns.tolist()
cols_to_drop = set([c for c in EXCLUDE_COLS if c in df.columns])
numeric_features = [c for c in num_cols if c not in cols_to_drop and c != TARGET_COL]

print(f"\nNumerische Features (exkl. Zeitspalten & Target): {len(numeric_features)}")
print(numeric_features[:20])

# --- Visuals ---
if isinstance(df.index, pd.DatetimeIndex) and TARGET_COL in df.columns:
    plt.figure(figsize=(12,4))
    df[TARGET_COL].plot()
    plt.title("Zielvariable über Zeit")
    plt.xlabel("Zeit")
    plt.ylabel(TARGET_COL)
    plt.show()

if TARGET_COL in df.columns:
    plt.figure(figsize=(6,4))
    df[TARGET_COL].plot(kind="hist", bins=50)
    plt.title("Histogramm Zielvariable")
    plt.xlabel(TARGET_COL)
    plt.show()

corr_cols = [TARGET_COL] + numeric_features[:min(25, len(numeric_features))]
corr = df[corr_cols].corr(numeric_only=True)
plt.figure(figsize=(10,8))
im = plt.imshow(corr.values, aspect='auto')
plt.colorbar(im, fraction=0.046, pad=0.04)
plt.xticks(range(len(corr_cols)), corr_cols, rotation=90)
plt.yticks(range(len(corr_cols)), corr_cols)
plt.title("Korrelationsmatrix (Subset)")
plt.tight_layout()
plt.show()


## 2) Supervised Datensatz & Metriken

In [ ]:

def make_supervised(df, features, target, lags, horizon):
    data = df[features + [target]].dropna().copy()
    X_seq, y_seq = [], []
    values = data.values
    F = len(features)
    for i in range(lags, len(data) - horizon + 1):
        past = values[i-lags:i, :F]
        future = values[i:i+horizon, F]
        X_seq.append(past)
        y_seq.append(future)
    X3d = np.array(X_seq)                 # (N, lags, F)
    Y = np.array(y_seq)                   # (N, horizon)
    X2d = X3d.reshape(X3d.shape[0], -1)   # (N, lags*F)
    return X3d, X2d, Y

def split_train_test(X, y, frac):
    n = len(X); s = int(n*frac)
    return X[:s], y[:s], X[s:], y[s:]

def eval_metrics(y_true, y_pred):
    y_true = np.asarray(y_true).reshape(len(y_true), -1)
    y_pred = np.asarray(y_pred).reshape(len(y_pred), -1)
    mae = mean_absolute_error(y_true, y_pred)
    mse = mean_squared_error(y_true, y_pred)
    rmse = float(np.sqrt(mse))
    r2 = r2_score(y_true, y_pred)
    return dict(mae=mae, rmse=rmse, mse=mse, r2=r2)


## 3) Supervised-Setup & Splits

In [ ]:

assert TARGET_COL in df.columns, "Target column not found in data."
X3d, X2d, Y = make_supervised(df, numeric_features, TARGET_COL, LAGS, HORIZON)
X3d_tr, Y_tr, X3d_te, Y_te = split_train_test(X3d, Y, TRAIN_FRACTION)
X2d_tr, _,   X2d_te, _   = split_train_test(X2d, Y, TRAIN_FRACTION)
print("Shapes -> X3d:", X3d.shape, " | X2d:", X2d.shape, " | Y:", Y.shape)
print("Train sizes:", X3d_tr.shape, Y_tr.shape, " | Test sizes:", X3d_te.shape, Y_te.shape)


## 4) Random Search Utilities

In [ ]:

def sample_from_space(space):
    out = {}
    for k, spec in space.items():
        t = spec.get("type")
        if t == "fixed":
            out[k] = spec["value"]
        elif t == "int":
            out[k] = int(np.random.randint(spec["min"], spec["max"] + 1))
        elif t == "float":
            out[k] = float(np.random.uniform(spec["min"], spec["max"]))
        elif t == "log_float":
            lo, hi = math.log(spec["min"]), math.log(spec["max"])
            out[k] = float(math.exp(np.random.uniform(lo, hi)))
        elif t == "int_choice":
            out[k] = int(np.random.choice(spec["choices"]))
        elif t == "choice":
            out[k] = random.choice(spec["choices"])
        elif t == "int_or_none":
            val = int(np.random.randint(spec["min"], spec["max"] + 1))
            out[k] = None if np.random.rand() < 0.1 else val
        else:
            raise ValueError(f"Unsupported type: {t}")
    return out

def save_final_config(filename, base_meta, best_params, search_keys):
    payload = {**base_meta}
    for k in search_keys:
        if k in best_params:
            payload[k] = best_params[k]
    with open(filename, "w", encoding="utf-8") as f:
        json.dump(payload, f, indent=2, ensure_ascii=False)
    print("[saved]", filename)


In [15]:
# --- Lag-weise (signal, lag) Permutation Importance auf dem TEST-Set ---

import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Falls die Spaltenliste noch nicht vorhanden ist: sauber herstellen
if isinstance(X2d_srv, pd.DataFrame):
    srv_cols = list(X2d_srv.columns)
else:
    srv_cols = [f"c{i}" for i in range(X2d_srv.shape[1])]
    X2d_srv = pd.DataFrame(X2d_srv, columns=srv_cols)

# Test-Set als DataFrame (nur hier wird permutiert!)
X2d_srv_te_df = pd.DataFrame(X2d_srv_te, columns=srv_cols)

rng = np.random.default_rng(123)

def mask_signal_lag(columns, signal: str, lag: int):
    """
    Wähle ALLE Spalten, die:
      - mit dem Basissignal beginnen (inkl. FE-Derivate wie 'signal_roll5_mean...')
      - und genau den gewünschten Lag tragen (Suffix '|t-<lag>')
    """
    suffix = f"|t-{lag}"
    out = []
    for i, c in enumerate(columns):
        cs = str(c)
        # Starts with original signal (FE-Derivate beginnen mit 'signal_...')
        if cs.startswith(signal) and cs.endswith(suffix):
            out.append(i)
    return out

def eval_metrics(y_true, y_pred):
    from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
    y_true = np.asarray(y_true).reshape(len(y_true), -1)
    y_pred = np.asarray(y_pred).reshape(len(y_pred), -1)
    mae = mean_absolute_error(y_true, y_pred)
    mse = mean_squared_error(y_true, y_pred)
    rmse = float(np.sqrt(mse))
    r2 = r2_score(y_true, y_pred)
    return dict(mae=mae, rmse=rmse, mse=mse, r2=r2)

def eval_metrics_per_h(y_true, y_pred):
    from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
    y_true = np.asarray(y_true).reshape(len(y_true), -1)
    y_pred = np.asarray(y_pred).reshape(len(y_pred), -1)
    H = y_true.shape[1]
    rows = []
    for h in range(H):
        mse_h = mean_squared_error(y_true[:, h], y_pred[:, h])
        rows.append(dict(
            h=h+1,
            mae=mean_absolute_error(y_true[:, h], y_pred[:, h]),
            rmse=float(np.sqrt(mse_h)),
            mse=mse_h,
            r2=r2_score(y_true[:, h], y_pred[:, h]),
        ))
    return pd.DataFrame(rows)

# Baseline-Metriken (ohne Permutation) sind bereits als `base` vorhanden:
# base = eval_metrics(Y_srv_te, rf_imp.predict(X2d_srv_te))

lagwise_rows = []
lagwise_rows_per_h = []  # optional: pro Horizont

for sig in base_server_candidates:
    for lag in range(LAGS, 0, -1):  # t-LAGS ... t-1
        idxs = mask_signal_lag(srv_cols, sig, lag)
        if not idxs:
            continue

        # Nur TEST-Set permutieren (die ausgewählten Lag-Spalten dieses Signals)
        Xp = X2d_srv_te_df.copy()
        for j in idxs:
            Xp.iloc[:, j] = rng.permutation(Xp.iloc[:, j].values)

        # Vorhersage & Metriken
        pred = rf_imp.predict(Xp.values)
        m_all = eval_metrics(Y_srv_te, pred)
        rmse_increase = m_all["rmse"] - base["rmse"]
        rel_percent = (rmse_increase / base["rmse"] * 100.0) if base["rmse"] > 0 else 0.0
        r2_drop = base["r2"] - m_all["r2"]

        lagwise_rows.append(dict(
            signal=sig, lag=lag,
            rmse_increase=rmse_increase,
            rel_percent=rel_percent,
            r2_drop=r2_drop
        ))

        # OPTIONAL: pro-Horizont aufschlüsseln
        df_h = eval_metrics_per_h(Y_srv_te, pred)
        for _, r in df_h.iterrows():
            lagwise_rows_per_h.append(dict(
                signal=sig, lag=lag, h=int(r["h"]),
                rmse=r["rmse"], r2=r["r2"]
            ))

# Tabelle 1: Lag-Importance je Signal
lag_imp_df = pd.DataFrame(lagwise_rows).sort_values(["signal", "lag"], ascending=[True, False])
display(lag_imp_df)
lag_imp_df.to_csv("Server_GroupPermutation_Lagwise_Importance.csv", index=False)
print("[Saved] Server_GroupPermutation_Lagwise_Importance.csv")

# (Optional) Tabelle 2: pro-Horizont (nur anschauen, wenn gebraucht)
if lagwise_rows_per_h:
    lag_imp_h_df = pd.DataFrame(lagwise_rows_per_h)
    # Beispiel: RMSE Heatmap je (signal, lag) aggregiert über alle Horizonte
    pivot_rmse = lag_imp_h_df.groupby(["signal","lag"])["rmse"].mean().unstack("lag").sort_index()
    print("\\nRMSE (Ø über Horizonte) je (Signal x Lag):")
    display(pivot_rmse)
    pivot_rmse.to_csv("Server_Lagwise_RMSE_by_Signal.csv")
    print("[Saved] Server_Lagwise_RMSE_by_Signal.csv")

    # Kleines Heatmap-Plot (optional)
    try:
        plt.figure(figsize=(10, 4 + 0.2*len(pivot_rmse)))
        im = plt.imshow(pivot_rmse.values, aspect="auto")
        plt.yticks(range(len(pivot_rmse.index)), pivot_rmse.index)
        plt.xticks(range(pivot_rmse.shape[1]), [f"t-{l}" for l in pivot_rmse.columns])
        plt.colorbar(im, label="RMSE (ø über Horizonte)")
        plt.title("Lag-wise Importance (RMSE ↑) – Server-Profil")
        plt.xlabel("Lag"); plt.ylabel("Signal")
        plt.tight_layout(); plt.show()
    except Exception as e:
        print("Plot skipped:", e)


NameError: name 'X2d_srv' is not defined

## 5) Config-Dateien (edge/server) erzeugen (falls fehlend)

In [ ]:

BASE = {
    "loading_strategy": "split",
    "train_fraction": 0.8,
    "validation_fraction": 0.2,
    "dataset": DATA_PATH,
    "target_column": TARGET_COL,
    "exclude_columns": EXCLUDE_COLS,
    "lags": LAGS,
    "horizon": HORIZON,
    "base_features": [TARGET_COL],
    "optimize_on": "server",
}

# --- RF ---
RF_EDGE = {
    "n_estimators": {"type": "int", "min": 50, "max": 250},
    "max_depth": {"type": "int_or_none", "min": 5, "max": 20},
    "min_samples_split": {"type": "int", "min": 2, "max": 10},
    "min_samples_leaf": {"type": "int", "min": 1, "max": 8},
    "max_features": {"type": "float", "min": 0.3, "max": 0.9},
    "bootstrap": {"type": "choice", "choices": [True, False]},
    "n_jobs": {"type": "fixed", "value": -1},
    "random_state": {"type": "fixed", "value": 42},
}
RF_SERVER = {
    "n_estimators": {"type": "int", "min": 200, "max": 1000},
    "max_depth": {"type": "int_or_none", "min": 10, "max": 40},
    "min_samples_split": {"type": "int", "min": 2, "max": 20},
    "min_samples_leaf": {"type": "int", "min": 1, "max": 10},
    "max_features": {"type": "float", "min": 0.3, "max": 1.0},
    "bootstrap": {"type": "choice", "choices": [True, False]},
    "n_jobs": {"type": "fixed", "value": -1},
    "random_state": {"type": "fixed", "value": 42},
}

# --- LSTM ---
LSTM_EDGE = {
    "num_layers": {"type": "fixed", "value": 1},
    "initial_units": {"type": "int", "min": 18, "max": 64},
    "dropout": {"type": "float", "min": 0.0, "max": 0.35},
    "batch_size": {"type": "int_choice", "choices": [16, 24, 32, 48, 64]},
    "epochs": {"type": "int", "min": 20, "max": 60},
    "learning_rate": {"type": "log_float", "min": 1e-4, "max": 5e-3},
    "loss": {"type": "choice", "choices": ["mse", "mae", "huber"]},
    "optimizer": {"type": "choice", "choices": ["adam", "rmsprop", "nadam"]},
    "clipnorm": {"type": "float", "min": 0.5, "max": 3.0},
}
LSTM_SERVER = {
    "num_layers": {"type": "int", "min": 2, "max": 3},
    "initial_units": {"type": "int", "min": 56, "max": 128},
    "dropout": {"type": "float", "min": 0.05, "max": 0.45},
    "batch_size": {"type": "int_choice", "choices": [32, 48, 64, 96, 128]},
    "epochs": {"type": "int", "min": 50, "max": 120},
    "learning_rate": {"type": "log_float", "min": 1e-4, "max": 3e-3},
    "loss": {"type": "choice", "choices": ["mse", "mae", "huber"]},
    "optimizer": {"type": "choice", "choices": ["adam", "rmsprop", "nadam", "adamw"]},
    "clipnorm": {"type": "float", "min": 0.5, "max": 5.0},
    "weight_decay": {"type": "log_float", "min": 1e-6, "max": 1e-3},
}

# --- XGB ---
XGB_EDGE = {
    "n_estimators": {"type": "int", "min": 100, "max": 400},
    "max_depth": {"type": "int", "min": 3, "max": 8},
    "learning_rate": {"type": "log_float", "min": 0.01, "max": 0.2},
    "subsample": {"type": "float", "min": 0.6, "max": 1.0},
    "colsample_bytree": {"type": "float", "min": 0.6, "max": 1.0},
    "min_child_weight": {"type": "int", "min": 1, "max": 8},
    "gamma": {"type": "float", "min": 0.0, "max": 5.0},
    "reg_lambda": {"type": "log_float", "min": 1e-3, "max": 10.0},
    "reg_alpha": {"type": "log_float", "min": 1e-6, "max": 1e-1},
    "tree_method": {"type": "fixed", "value": "hist"},
    "n_jobs": {"type": "fixed", "value": -1},
    "random_state": {"type": "fixed", "value": 42},
}
XGB_SERVER = {
    "n_estimators": {"type": "int", "min": 400, "max": 1500},
    "max_depth": {"type": "int", "min": 6, "max": 12},
    "learning_rate": {"type": "log_float", "min": 0.01, "max": 0.1},
    "subsample": {"type": "float", "min": 0.6, "max": 1.0},
    "colsample_bytree": {"type": "float", "min": 0.6, "max": 1.0},
    "min_child_weight": {"type": "int", "min": 1, "max": 12},
    "gamma": {"type": "float", "min": 0.0, "max": 8.0},
    "reg_lambda": {"type": "log_float", "min": 1e-3, "max": 100.0},
    "reg_alpha": {"type": "log_float", "min": 1e-6, "max": 1.0},
    "tree_method": {"type": "fixed", "value": "hist"},
    "n_jobs": {"type": "fixed", "value": -1},
    "random_state": {"type": "fixed", "value": 42},
}

# --- LGBM ---
LGBM_EDGE = {
    "n_estimators": {"type": "int", "min": 200, "max": 1000},
    "learning_rate": {"type": "log_float", "min": 0.01, "max": 0.2},
    "num_leaves": {"type": "int", "min": 16, "max": 64},
    "min_child_samples": {"type": "int", "min": 5, "max": 50},
    "subsample": {"type": "float", "min": 0.6, "max": 1.0},
    "colsample_bytree": {"type": "float", "min": 0.6, "max": 1.0},
    "reg_alpha": {"type": "log_float", "min": 1e-6, "max": 1e-1},
    "reg_lambda": {"type": "log_float", "min": 1e-3, "max": 10.0},
    "max_bin": {"type": "int", "min": 63, "max": 255},
}
LGBM_SERVER = {
    "n_estimators": {"type": "int", "min": 500, "max": 3000},
    "learning_rate": {"type": "log_float", "min": 0.01, "max": 0.1},
    "num_leaves": {"type": "int", "min": 31, "max": 255},
    "min_child_samples": {"type": "int", "min": 5, "max": 100},
    "subsample": {"type": "float", "min": 0.5, "max": 1.0},
    "colsample_bytree": {"type": "float", "min": 0.5, "max": 1.0},
    "reg_alpha": {"type": "log_float", "min": 1e-6, "max": 1.0},
    "reg_lambda": {"type": "log_float", "min": 1e-3, "max": 100.0},
    "max_bin": {"type": "int", "min": 127, "max": 511},
}

# --- CNN1D ---
CNN_EDGE = {
    "cnn_blocks": {"type": "int", "min": 1, "max": 2},
    "cnn_base_filters": {"type": "int", "min": 16, "max": 64},
    "cnn_kernel_size": {"type": "int", "min": 3, "max": 7},
    "cnn_dropout": {"type": "float", "min": 0.0, "max": 0.35},
    "cnn_activation": {"type": "choice", "choices": ["relu", "gelu"]},
    "batch_size": {"type": "int_choice", "choices": [16, 32, 48, 64]},
    "epochs": {"type": "int", "min": 20, "max": 60},
    "optimizer": {"type": "choice", "choices": ["adam", "rmsprop", "nadam"]},
    "learning_rate": {"type": "log_float", "min": 1e-4, "max": 5e-3},
    "clipnorm": {"type": "float", "min": 0.5, "max": 3.0},
}
CNN_SERVER = {
    "cnn_blocks": {"type": "int", "min": 2, "max": 4},
    "cnn_base_filters": {"type": "int", "min": 64, "max": 256},
    "cnn_kernel_size": {"type": "int", "min": 3, "max": 9},
    "cnn_dropout": {"type": "float", "min": 0.05, "max": 0.5},
    "cnn_activation": {"type": "choice", "choices": ["relu", "gelu"]},
    "batch_size": {"type": "int_choice", "choices": [32, 64, 96, 128]},
    "epochs": {"type": "int", "min": 50, "max": 150},
    "optimizer": {"type": "choice", "choices": ["adam", "rmsprop", "nadam", "adamw"]},
    "learning_rate": {"type": "log_float", "min": 1e-4, "max": 3e-3},
    "clipnorm": {"type": "float", "min": 0.5, "max": 5.0},
    "weight_decay": {"type": "log_float", "min": 1e-6, "max": 1e-3},
}

CONFIG_TEMPLATES = {
    "config_rf_edge_opt.json":    {**BASE, "model_name":"random_forest_edge_opt",  "deployment_profile":"edge",   "search_space":RF_EDGE},
    "config_rf_server_opt.json":  {**BASE, "model_name":"random_forest_server_opt","deployment_profile":"server", "search_space":RF_SERVER},
    "config_lstm_edge_opt.json":  {**BASE, "model_name":"lstm_edge_opt",           "deployment_profile":"edge",   "search_space":LSTM_EDGE},
    "config_lstm_server_opt.json":{**BASE, "model_name":"lstm_server_opt",         "deployment_profile":"server", "search_space":LSTM_SERVER},
    "config_xgb_edge_opt.json":   {**BASE, "model_name":"xgboost_edge_opt",        "deployment_profile":"edge",   "search_space":XGB_EDGE},
    "config_xgb_server_opt.json": {**BASE, "model_name":"xgboost_server_opt",      "deployment_profile":"server", "search_space":XGB_SERVER},
    "config_lgbm_edge_opt.json":  {**BASE, "model_name":"lightgbm_edge_opt",       "deployment_profile":"edge",   "search_space":LGBM_EDGE},
    "config_lgbm_server_opt.json":{**BASE, "model_name":"lightgbm_server_opt",     "deployment_profile":"server", "search_space":LGBM_SERVER},
    "config_cnn1d_edge_opt.json": {**BASE, "model_name":"cnn1d_edge_opt",          "deployment_profile":"edge",   "search_space":CNN_EDGE},
    "config_cnn1d_server_opt.json":{**BASE,"model_name":"cnn1d_server_opt",        "deployment_profile":"server", "search_space":CNN_SERVER},
}

for fname, payload in CONFIG_TEMPLATES.items():
    if not os.path.exists(fname):
        with open(fname, "w", encoding="utf-8") as f:
            json.dump(payload, f, indent=2, ensure_ascii=False)
        print("[created]", fname)
    else:
        print("[exists ]", fname)


## 6) Random Forest – Random Search (Edge & Server)

In [ ]:

from sklearn.ensemble import RandomForestRegressor
import pandas as pd

with open("config_rf_edge_opt.json","r",encoding="utf-8") as f:
    rf_edge_cfg = json.load(f)
with open("config_rf_server_opt.json","r",encoding="utf-8") as f:
    rf_server_cfg = json.load(f)

def rf_random_search(space, trials=40):
    results, best = [], None
    for _ in range(trials):
        p = sample_from_space(space)
        rf = RandomForestRegressor(**{k:p[k] for k in p if k in [
            "n_estimators","max_depth","min_samples_split","min_samples_leaf","max_features","bootstrap","random_state","n_jobs"
        ]})
        rf.fit(X2d_tr, Y_tr)
        pred = rf.predict(X2d_te)
        m = eval_metrics(Y_te, pred)
        row = {**p, **m}; results.append(row)
        if best is None or m["rmse"] < best["rmse"]:
            best = row
    return pd.DataFrame(results), best

df_rf_edge, best_rf_edge = rf_random_search(rf_edge_cfg["search_space"], trials=30)
df_rf_server, best_rf_server = rf_random_search(rf_server_cfg["search_space"], trials=50)

display_dataframe_to_user("RF_Edge_Search_Results", df_rf_edge.sort_values("rmse").head(20))
display_dataframe_to_user("RF_Server_Search_Results", df_rf_server.sort_values("rmse").head(20))

save_final_config("rf_edge_final_config.json",
    {"model_name":"random_forest","deployment_profile":"edge","dataset":DATA_PATH,"lags":LAGS,"horizon":HORIZON},
    best_rf_edge, rf_edge_cfg["search_space"].keys()
)
save_final_config("rf_server_final_config.json",
    {"model_name":"random_forest","deployment_profile":"server","dataset":DATA_PATH,"lags":LAGS,"horizon":HORIZON},
    best_rf_server, rf_server_cfg["search_space"].keys()
)

print("Best RF Edge:", best_rf_edge)
print("Best RF Server:", best_rf_server)


## 7) LSTM – Random Search (Edge & Server)

In [ ]:

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout, BatchNormalization
from tensorflow.keras.optimizers import Adam, RMSprop, Nadam
from sklearn.preprocessing import MinMaxScaler
import pandas as pd

# Scale X3d/Y
X3d_tr_ = X3d_tr.reshape(X3d_tr.shape[0], -1)
X3d_te_ = X3d_te.reshape(X3d_te.shape[0], -1)
x_scaler = MinMaxScaler().fit(X3d_tr_)
X3d_tr_s = x_scaler.transform(X3d_tr_).reshape(X3d_tr.shape)
X3d_te_s = x_scaler.transform(X3d_te_).reshape(X3d_te.shape)
y_scaler = MinMaxScaler().fit(Y_tr)
Y_tr_s = y_scaler.transform(Y_tr)
Y_te_s = y_scaler.transform(Y_te)

with open("config_lstm_edge_opt.json","r",encoding="utf-8") as f:
    lstm_edge_cfg = json.load(f)
with open("config_lstm_server_opt.json","r",encoding="utf-8") as f:
    lstm_server_cfg = json.load(f)

def build_lstm(input_shape, num_layers=1, initial_units=64, dropout=0.1, horizon=HORIZON):
    m = Sequential()
    units = int(initial_units)
    for i in range(num_layers):
        m.add(LSTM(units, return_sequences=(i < num_layers-1), input_shape=input_shape if i==0 else None))
        m.add(Dropout(dropout))
        m.add(BatchNormalization())
        units = max(units // 2, 4)
    m.add(Dense(horizon, activation="linear"))
    return m

def make_optimizer(p):
    name = str(p.get("optimizer","adam")).lower()
    lr = float(p.get("learning_rate", 1e-3))
    clipnorm = float(p.get("clipnorm", 0.0)) if p.get("clipnorm", 0.0) else None
    kw = {"learning_rate": lr}
    if clipnorm and clipnorm>0: kw["clipnorm"] = clipnorm
    if name=="adam":   return Adam(**kw)
    if name=="rmsprop":return RMSprop(**kw)
    if name=="nadam":  return Nadam(**kw)
    if name=="adamw":
        try: return tf.keras.optimizers.AdamW(weight_decay=float(p.get("weight_decay",0.0)), **kw)
        except: return Adam(**kw)
    return Adam(**kw)

def lstm_random_search(space, trials=20):
    results, best = [], None
    input_shape = (LAGS, len(numeric_features))
    for _ in range(trials):
        p = sample_from_space(space)
        model = build_lstm(input_shape, p.get("num_layers",1), p.get("initial_units",64), p.get("dropout",0.1), HORIZON)
        opt = make_optimizer(p)
        model.compile(optimizer=opt, loss=p.get("loss","mse"), metrics=["mae"])
        split = int(len(X3d_tr_s) * (1 - VAL_FRACTION))
        X_fit, X_val = X3d_tr_s[:split], X3d_tr_s[split:]
        y_fit, y_val = Y_tr_s[:split], Y_tr_s[split:]
        cb=[tf.keras.callbacks.EarlyStopping(monitor="val_loss", patience=10, restore_best_weights=True)]
        model.fit(X_fit, y_fit,
                  validation_data=(X_val, y_val) if len(X_val)>0 else None,
                  epochs=int(p.get("epochs",40)), batch_size=int(p.get("batch_size",32)),
                  verbose=0, callbacks=cb)
        pred_s = model.predict(X3d_te_s, verbose=0)
        pred = y_scaler.inverse_transform(pred_s)
        m = eval_metrics(Y_te, pred)
        row = {**p, **m}; results.append(row)
        if best is None or m["rmse"] < best["rmse"]:
            best = row
    return pd.DataFrame(results), best

df_lstm_edge, best_lstm_edge = lstm_random_search(lstm_edge_cfg["search_space"], trials=20)
df_lstm_server, best_lstm_server = lstm_random_search(lstm_server_cfg["search_space"], trials=30)

display_dataframe_to_user("LSTM_Edge_Search_Results", df_lstm_edge.sort_values("rmse").head(20))
display_dataframe_to_user("LSTM_Server_Search_Results", df_lstm_server.sort_values("rmse").head(20))

save_final_config("lstm_edge_final_config.json",
    {"model_name":"lstm","deployment_profile":"edge","dataset":DATA_PATH,"lags":LAGS,"horizon":HORIZON},
    best_lstm_edge, lstm_edge_cfg["search_space"].keys()
)
save_final_config("lstm_server_final_config.json",
    {"model_name":"lstm","deployment_profile":"server","dataset":DATA_PATH,"lags":LAGS,"horizon":HORIZON},
    best_lstm_server, lstm_server_cfg["search_space"].keys()
)

print("Best LSTM Edge:", best_lstm_edge)
print("Best LSTM Server:", best_lstm_server)


## 8) XGBoost – Random Search (Edge & Server)

In [ ]:

from sklearn.multioutput import MultiOutputRegressor
from xgboost import XGBRegressor
import pandas as pd

with open("config_xgb_edge_opt.json","r",encoding="utf-8") as f:
    xgb_edge_cfg = json.load(f)
with open("config_xgb_server_opt.json","r",encoding="utf-8") as f:
    xgb_server_cfg = json.load(f)

def xgb_random_search(space, trials=40):
    results, best = [], None
    for _ in range(trials):
        p = sample_from_space(space)
        base = XGBRegressor(
            n_estimators=int(p["n_estimators"]),
            max_depth=int(p["max_depth"]),
            learning_rate=float(p["learning_rate"]),
            subsample=float(p["subsample"]),
            colsample_bytree=float(p["colsample_bytree"]),
            min_child_weight=int(p["min_child_weight"]),
            gamma=float(p["gamma"]),
            reg_lambda=float(p["reg_lambda"]),
            reg_alpha=float(p["reg_alpha"]),
            tree_method=p["tree_method"],
            n_jobs=p["n_jobs"],
            random_state=p["random_state"],
            objective="reg:squarederror",
        )
        model = MultiOutputRegressor(base)
        model.fit(X2d_tr, Y_tr)
        pred = model.predict(X2d_te)
        m = eval_metrics(Y_te, pred)
        row = {**p, **m}; results.append(row)
        if best is None or m["rmse"] < best["rmse"]:
            best = row
    return pd.DataFrame(results), best

df_xgb_edge, best_xgb_edge = xgb_random_search(xgb_edge_cfg["search_space"], trials=40)
df_xgb_server, best_xgb_server = xgb_random_search(xgb_server_cfg["search_space"], trials=60)

display_dataframe_to_user("XGB_Edge_Search_Results", df_xgb_edge.sort_values("rmse").head(20))
display_dataframe_to_user("XGB_Server_Search_Results", df_xgb_server.sort_values("rmse").head(20))

save_final_config("xgb_edge_final_config.json",
    {"model_name":"xgboost","deployment_profile":"edge","dataset":DATA_PATH,"lags":LAGS,"horizon":HORIZON},
    best_xgb_edge, xgb_edge_cfg["search_space"].keys()
)
save_final_config("xgb_server_final_config.json",
    {"model_name":"xgboost","deployment_profile":"server","dataset":DATA_PATH,"lags":LAGS,"horizon":HORIZON},
    best_xgb_server, xgb_server_cfg["search_space"].keys()
)

print("Best XGB Edge:", best_xgb_edge)
print("Best XGB Server:", best_xgb_server)


## 9) LightGBM – Random Search (Edge & Server)

In [ ]:

from sklearn.multioutput import MultiOutputRegressor
import pandas as pd

with open("config_lgbm_edge_opt.json","r",encoding="utf-8") as f:
    lgbm_edge_cfg = json.load(f)
with open("config_lgbm_server_opt.json","r",encoding="utf-8") as f:
    lgbm_server_cfg = json.load(f)

def lgbm_random_search(space, trials=40):
    from lightgbm import LGBMRegressor
    results, best = [], None
    for _ in range(trials):
        p = sample_from_space(space)
        base = LGBMRegressor(
            n_estimators=int(p["n_estimators"]),
            learning_rate=float(p["learning_rate"]),
            num_leaves=int(p["num_leaves"]),
            min_child_samples=int(p["min_child_samples"]),
            subsample=float(p["subsample"]),
            colsample_bytree=float(p["colsample_bytree"]),
            reg_alpha=float(p["reg_alpha"]),
            reg_lambda=float(p["reg_lambda"]),
            max_bin=int(p["max_bin"]),
            random_state=42,
            n_jobs=-1
        )
        model = MultiOutputRegressor(base)
        model.fit(X2d_tr, Y_tr)
        pred = model.predict(X2d_te)
        m = eval_metrics(Y_te, pred)
        row = {**p, **m}; results.append(row)
        if best is None or m["rmse"] < best["rmse"]:
            best = row
    return pd.DataFrame(results), best

df_lgbm_edge, best_lgbm_edge = lgbm_random_search(lgbm_edge_cfg["search_space"], trials=40)
df_lgbm_server, best_lgbm_server = lgbm_random_search(lgbm_server_cfg["search_space"], trials=60)

display_dataframe_to_user("LGBM_Edge_Search_Results", df_lgbm_edge.sort_values("rmse").head(20))
display_dataframe_to_user("LGBM_Server_Search_Results", df_lgbm_server.sort_values("rmse").head(20))

save_final_config("lgbm_edge_final_config.json",
    {"model_name":"lightgbm","deployment_profile":"edge","dataset":DATA_PATH,"lags":LAGS,"horizon":HORIZON},
    best_lgbm_edge, lgbm_edge_cfg["search_space"].keys()
)
save_final_config("lgbm_server_final_config.json",
    {"model_name":"lightgbm","deployment_profile":"server","dataset":DATA_PATH,"lags":LAGS,"horizon":HORIZON},
    best_lgbm_server, lgbm_server_cfg["search_space"].keys()
)

print("Best LGBM Edge:", best_lgbm_edge)
print("Best LGBM Server:", best_lgbm_server)


## 10) CNN1D – Random Search (Edge & Server)

In [ ]:

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv1D, BatchNormalization, Activation, Dropout, GlobalAveragePooling1D, Dense, Input
from tensorflow.keras.optimizers import Adam, RMSprop, Nadam
from sklearn.preprocessing import MinMaxScaler
import pandas as pd

with open("config_cnn1d_edge_opt.json","r",encoding="utf-8") as f:
    cnn_edge_cfg = json.load(f)
with open("config_cnn1d_server_opt.json","r",encoding="utf-8") as f:
    cnn_server_cfg = json.load(f)

# Scaling (wie LSTM)
X3d_tr_ = X3d_tr.reshape(X3d_tr.shape[0], -1)
X3d_te_ = X3d_te.reshape(X3d_te.shape[0], -1)
x_scaler_c = MinMaxScaler().fit(X3d_tr_)
X3d_tr_s = x_scaler_c.transform(X3d_tr_).reshape(X3d_tr.shape)
X3d_te_s = x_scaler_c.transform(X3d_te_).reshape(X3d_te.shape)
y_scaler_c = MinMaxScaler().fit(Y_tr)
Y_tr_s = y_scaler_c.transform(Y_tr)
Y_te_s = y_scaler_c.transform(Y_te)

def build_cnn1d(input_shape, blocks=2, base_filters=64, kernel_size=5, dropout=0.1, activation="relu", horizon=HORIZON):
    m = Sequential([Input(shape=input_shape)])
    filters = int(base_filters)
    for i in range(int(blocks)):
        m.add(Conv1D(filters=filters, kernel_size=int(kernel_size), padding="same"))
        m.add(BatchNormalization()); m.add(Activation(activation))
        if dropout and dropout>0: m.add(Dropout(float(dropout)))
        filters = max(filters // 2, 4)
    m.add(GlobalAveragePooling1D())
    m.add(Dense(max(filters, 8))); m.add(Activation(activation))
    m.add(Dense(horizon, activation="linear"))
    return m

def make_optimizer(p):
    name = str(p.get("optimizer","adam")).lower()
    lr = float(p.get("learning_rate", 1e-3))
    clipnorm = float(p.get("clipnorm", 0.0)) if p.get("clipnorm", 0.0) else None
    kw = {"learning_rate": lr}
    if clipnorm and clipnorm>0: kw["clipnorm"] = clipnorm
    if name=="adam": return Adam(**kw)
    if name=="rmsprop": return RMSprop(**kw)
    if name=="nadam": return Nadam(**kw)
    if name=="adamw":
        try: return tf.keras.optimizers.AdamW(weight_decay=float(p.get("weight_decay",0.0)), **kw)
        except: return Adam(**kw)
    return Adam(**kw)

def cnn_random_search(space, trials=20):
    results, best = [], None
    input_shape = (LAGS, len(numeric_features))
    for _ in range(trials):
        p = sample_from_space(space)
        model = build_cnn1d(input_shape,
                            p.get("cnn_blocks",2), p.get("cnn_base_filters",64),
                            p.get("cnn_kernel_size",5), p.get("cnn_dropout",0.1),
                            p.get("cnn_activation","relu"), HORIZON)
        opt = make_optimizer(p)
        model.compile(optimizer=opt, loss="huber", metrics=["mae"])
        split = int(len(X3d_tr_s) * (1 - VAL_FRACTION))
        X_fit, X_val = X3d_tr_s[:split], X3d_tr_s[split:]
        y_fit, y_val = Y_tr_s[:split], Y_tr_s[split:]
        cb=[tf.keras.callbacks.EarlyStopping(monitor="val_loss", patience=10, restore_best_weights=True)]
        model.fit(X_fit, y_fit,
                  validation_data=(X_val, y_val) if len(X_val)>0 else None,
                  epochs=int(p.get("epochs",40)), batch_size=int(p.get("batch_size",32)),
                  verbose=0, callbacks=cb)
        pred_s = model.predict(X3d_te_s, verbose=0)
        pred = y_scaler_c.inverse_transform(pred_s)
        m = eval_metrics(Y_te, pred)
        row = {**p, **m}; results.append(row)
        if best is None or m["rmse"] < best["rmse"]:
            best = row
    return pd.DataFrame(results), best

df_cnn_edge, best_cnn_edge = cnn_random_search(cnn_edge_cfg["search_space"], trials=20)
df_cnn_server, best_cnn_server = cnn_random_search(cnn_server_cfg["search_space"], trials=30)

display_dataframe_to_user("CNN1D_Edge_Search_Results", df_cnn_edge.sort_values("rmse").head(20))
display_dataframe_to_user("CNN1D_Server_Search_Results", df_cnn_server.sort_values("rmse").head(20))

save_final_config("cnn1d_edge_final_config.json",
    {"model_name":"cnn1d","deployment_profile":"edge","dataset":DATA_PATH,"lags":LAGS,"horizon":HORIZON},
    best_cnn_edge, cnn_edge_cfg["search_space"].keys()
)
save_final_config("cnn1d_server_final_config.json",
    {"model_name":"cnn1d","deployment_profile":"server","dataset":DATA_PATH,"lags":LAGS,"horizon":HORIZON},
    best_cnn_server, cnn_server_cfg["search_space"].keys()
)

print("Best CNN Edge:", best_cnn_edge)
print("Best CNN Server:", best_cnn_server)
